# 09 - Validation adaptation and repeated central experiments

The same prespecified validation-only adaptation grid and selection rule is applied independently to every dataset before its locked test is opened. CSE-CIC-IDS2018 and Edge-IIoTset are development datasets; TabularIoTAttack-2024 and DataSense 2025 are untouched external confirmations.


In [1]:
from pathlib import Path
import os, sys

LOCAL_CANDIDATES = [
    Path('/Volumes/LocalDrive1/onlyscholar-projects/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
    Path('/content/drive/MyDrive/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION'),
]

def is_project_root(path: Path) -> bool:
    return (path / 'pyproject.toml').is_file() and (path / 'src' / 'rareguard').is_dir()

def find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents, *LOCAL_CANDIDATES, *DRIVE_CANDIDATES]:
        if is_project_root(candidate):
            return candidate

    if Path('/content').is_dir():
        from google.colab import drive

        drive.mount('/content/drive', force_remount=False)
        for candidate in DRIVE_CANDIDATES:
            if is_project_root(candidate):
                return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'RareGuard-XIDS IMPLEMENTATION project not found. Expected the canonical '
        'Google Drive project at '
        '/content/drive/MyDrive/projects/implementation/'
        'RareGuard-XIDS[DeepakSir]/IMPLEMENTATION or the canonical local project.'
    )
os.chdir(PROJECT_ROOT)
PROJECT_SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, PROJECT_SRC)
os.environ['PYTHONPATH'] = PROJECT_SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
print('Project root:', PROJECT_ROOT)
print('Runtime:', 'Colab' if Path('/content').is_dir() else 'local')
print('Python path ready')


Project root: /content/drive/MyDrive/projects/implementation/RareGuard-XIDS[DeepakSir]/IMPLEMENTATION
Runtime: Colab
Python path ready


In [2]:
TUNING_DEVICE = 'cpu'  # Accelerator use is restricted to validation-only adaptation.
!python -m rareguard.experiments.run_publication_adaptive_tuning --config configs/publication.yaml --datasets configs/datasets.yaml --dataset all --device {TUNING_DEVICE} --resume


2026-10-23 14:05:48,791 | INFO | rareguard | selected dataset=cse_cic_ids2018 validation_score=0.6308 settings=(0.500, 0.000, 0.100)
2026-10-23 14:05:51,952 | INFO | rareguard | selected dataset=cic_bccc_nrc_tabulariotattack_2024 validation_score=0.3648 settings=(0.500, 0.250, 0.100)
2026-10-23 14:05:54,041 | INFO | rareguard | selected dataset=cic_iiot_2025_datasense validation_score=0.8099 settings=(1.000, 0.000, 0.100)
2026-10-23 14:05:55,300 | INFO | rareguard | selected dataset=edge_iiotset validation_score=0.8906 settings=(0.500, 0.000, 0.100)


In [3]:
!python -m rareguard.experiments.run_publication_baselines --config configs/publication.yaml --datasets configs/datasets.yaml --dataset all --resume


In [4]:
CONFIRMATORY_DEVICE = 'cpu'
!python -m rareguard.experiments.run_publication_central --config configs/publication.yaml --datasets configs/datasets.yaml --dataset all --device {CONFIRMATORY_DEVICE} --resume


2026-10-23 14:06:08,217 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 NOVA-XIDS
2026-10-23 14:06:08,720 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 RareGuard-XIDS
2026-10-23 14:06:08,720 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 RareGuard-no-class-balance
2026-10-23 14:06:08,721 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 RareGuard-no-contrastive
2026-10-23 14:06:08,721 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 RareGuard-no-prototype
2026-10-23 14:06:08,721 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=42 RareGuard-no-direct-path
2026-10-23 14:06:08,722 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=73 NOVA-XIDS
2026-10-23 14:06:08,722 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=73 RareGuard-XIDS
2026-10-23 14:06:08,722 | INFO | rareguard | Skipping completed cse_cic_ids2018 seed=73 RareGuard-no-class-balance
2026-10-23 14:06:08,722 | INFO 

In [5]:
import pandas as pd
display(pd.read_csv('artifacts/publication/results/central_summary_ci.csv'))


,dataset,model,seeds,best_epoch_mean,best_epoch_ci95_half,best_validation_selection_score_mean,best_validation_selection_score_ci95_half,best_validation_f1_macro_mean,best_validation_f1_macro_ci95_half,best_validation_rare_f1_macro_mean,...,ece_15_mean,ece_15_ci95_half,rare_class_count_mean,rare_class_count_ci95_half,rare_f1_macro_mean,rare_f1_macro_ci95_half,roc_auc_macro_ovr_mean,roc_auc_macro_ovr_ci95_half,pr_auc_macro_mean,pr_auc_macro_ci95_half
0,cic_bccc_nrc_tabulariotattack_2024,NOVA-XIDS,5,16.000000,4.249179,0.355365,0.026332,0.575199,0.023553,0.135531,...,0.035922,0.007406,1.0,0.0,0.109217,0.025601,0.937544,0.003147,0.600060,0.010214
1,cic_bccc_nrc_tabulariotattack_2024,RareGuard-XIDS,5,17.400000,4.744658,0.356479,0.010746,0.598559,0.003973,0.114399,...,0.029153,0.005354,1.0,0.0,0.082213,0.023621,0.943793,0.001968,0.627333,0.007820
2,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-class-balance,3,22.333333,2.847814,0.306995,0.002709,0.613990,0.005419,0.000000,...,0.025086,0.009859,1.0,0.0,0.000000,0.000000,0.948142,0.001017,0.641380,0.004348
3,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-contrastive,3,22.000000,2.993949,0.355665,0.023846,0.610361,0.005585,0.100968,...,0.025694,0.008440,1.0,0.0,0.074004,0.033548,0.946019,0.001694,0.636432,0.002179
4,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-direct-path,3,12.333333,10.873634,0.350543,0.008214,0.556907,0.077764,0.144180,...,0.027394,0.010254,1.0,0.0,0.092812,0.030624,0.933067,0.014964,0.590490,0.052600
5,cic_bccc_nrc_tabulariotattack_2024,RareGuard-no-prototype,3,13.666667,8.642788,0.352597,0.015990,0.596493,0.015138,0.108700,...,0.024107,0.006083,1.0,0.0,0.091064,0.038803,0.943624,0.005866,0.623625,0.017168
6,cic_iiot_2025_datasense,NOVA-XIDS,5,20.800000,4.529254,0.566241,0.044722,0.603821,0.031626,0.528662,...,0.069012,0.010111,18.0,0.0,0.547295,0.040573,0.985961,0.001418,0.755341,0.027053
7,cic_iiot_2025_datasense,RareGuard-XIDS,5,21.600000,2.880600,0.808338,0.014851,0.801559,0.015923,0.815117,...,0.050774,0.009493,18.0,0.0,0.803393,0.026080,0.993179,0.000284,0.869484,0.014415
8,cic_iiot_2025_datasense,RareGuard-no-class-balance,3,19.333333,3.637606,0.810187,0.030745,0.808717,0.013431,0.811657,...,0.034373,0.012820,18.0,0.0,0.792246,0.028191,0.993840,0.000738,0.883755,0.016142
9,cic_iiot_2025_datasense,RareGuard-no-contrastive,3,21.333333,2.355627,0.774957,0.008896,0.781342,0.014219,0.768573,...,0.044395,0.007992,18.0,0.0,0.770419,0.021759,0.993701,0.000517,0.875067,0.015829
